# Introduction:
This notebook is used as the intial inspection point for potential datasets to be used for the sythentic datasset generation step, used after preprocessing is complete. The work completed in this datset does not contribute to the core functionality of the project. It has been left in as a display of background workt that had been completed. 

# Imports:
Below is the cell used to import any packages needed to run the subsequent cells. 

In [78]:
import pandas as pd

# Load Datasets:
The below cell is used as the loading point for the datasets that will be inspected. 

In [79]:
# LOAD DATASET ONE
df = pd.read_csv('../0_Data/raw_data/Online_Retail_One.csv', skiprows=1)
# LOAD DATASET TWO 
df2 = pd.read_csv('../0_Data/raw_data/Online_Retail_Two.csv', skiprows=1)
# LOAD DATASET THREE    
df3 = pd.read_csv('../0_Data/raw_data/Consumer_Meta.csv', skiprows=1)

# Dataset Inspection:
Dataset sourcing was split into two parts: 
- 1. Finding a dataset representing customer loyalty. Essentially, a dataset that displayed more numerous transactions per unique ID. This was done to enable reliable clustering results. A bench mark for this was at least 10:1 (Transactions per Customer)
- 2. A comprehensive dataset, detailing a business itself. For example, locations, products, prices etc. 

The objective set out for this dataset strategy was to merge suitable datasets together to create one comphrensive/profficient/data-rich dataset that can then be fed into SDV. 

In [80]:
print("----------------------------------")
print(" #### DATASET ONE INSPECTION #### ")
print("Dataset One Shape:", df.shape)
# Transaction per Unique ID Insepction: 
# Drop Null observations from Customer ID:
df = df.dropna(subset=['Customer ID'])
#Drop Duplicated observations: 
df = df.drop_duplicates()
# Display value of unique ID's
print("Unique ID's:", df['Customer ID'].nunique())
#Calcualte Transaction per ID ratio:
ratio = len(df) / df["Customer ID"].nunique()
print("Transaction to Unqiue ID Ratio:", round(ratio))
print("")
#Print Dataset Categories List:
print("Dataset Categories:")
#Use a for loop to print vertically (Readability)
for col in df.columns:
    print(col)

----------------------------------
 #### DATASET ONE INSPECTION #### 
Dataset One Shape: (525461, 8)
Unique ID's: 4383
Transaction to Unqiue ID Ratio: 94

Dataset Categories:
Invoice
StockCode
Description
Quantity
InvoiceDate
Price
Customer ID
Country


In [81]:
print("----------------------------------")
print(" #### DATASET TWO INSPECTION #### ")
print("Dataset Two Shape:", df2.shape)
# Transaction per Unique ID Insepction: 
# Drop Null observations from Customer ID:
df2 = df2.dropna(subset=['Customer ID'])
#Drop Duplicated observations: 
df2 = df2.drop_duplicates()
# Display value of unique ID's
print("Unique ID's:", df2['Customer ID'].nunique())
#Calcualte Transaction per ID ratio:
ratio = len(df2) / df2["Customer ID"].nunique()
print("Transaction to Unqiue ID Ratio:", round(ratio))
print("")
#Print Dataset Categories List:
print("Dataset Categories:")
#Use a for loop to print vertically (Readability)
for col in df2.columns:
    print(col)

----------------------------------
 #### DATASET TWO INSPECTION #### 
Dataset Two Shape: (541910, 8)
Unique ID's: 4372
Transaction to Unqiue ID Ratio: 92

Dataset Categories:
Invoice
StockCode
Description
Quantity
InvoiceDate
Price
Customer ID
Country


In [88]:
print("----------------------------------")
print(" #### DATASET THREE INSPECTION #### ")
print("Dataset Three Shape:", df3.shape)
# Transaction per Unique ID Inspection: 
#Drop any null observations from Customer ID:
df3 = df3.dropna(subset=['customer_id'])
df3 = df3.drop_duplicates()
#Display sun of unique ID's:
print("Unique ID's:", df3["customer_id"].nunique())
#Calculate Transaction per ID ratio: 
ratio = len(df3) / df3["customer_id"].nunique()
print("Transaction to Unique ID Ratio:", round(ratio, 3))
print("")
#Print Dataset Categories List:
print("Dataset Categories:")
#Use a for loop to print vertically (Readability)
for col in df3.columns:
    print(col)

----------------------------------
 #### DATASET THREE INSPECTION #### 
Dataset Three Shape: (20000, 20)
Unique ID's: 19250
Transaction to Unique ID Ratio: 1.039

Dataset Categories:
transaction_id
timestamp
store_id
city
country
store_type
product_category
product_name
unit_price
quantity
discount_applied
payment_method
customer_id
customer_age_group
customer_gender
loyalty_member
weather_condition
temperature_c
holiday_name
total_amount


# Duplicate Dataframe Inspection. 
The results from df and df2 show very simialar results. With df having a shape of (525641, 8) and df2 having a shape of (541910, 8). As well as both posessing similar tranaction to unique id ratios of 94 and 92 resectively, not to mention both dataframes having the same categories. The concern is that these datasets might have a considerable overlap between observations. Further investigation follows below. 

In [89]:
# Calculate 'overlap' - Checking for Customer ID overlaping values. 
overlap = set(df['Customer ID']) & set(df2['Customer ID'])
print(len(overlap))

2813


There are is around ~4370-4780 unique Customer ID in both datasets. The result from the overlapping check displays that 2813 ID's appear in both sets. This is 64% overlap. This now changes the approach in two ways. 

1. If combined, the new dataset will be able to provide a longer time window per cusotomer as the dataset now contains more repeat transactions per customer, strengthening the frequency. 
2. Dates now need to be checked in order to confirm wether the datsets contain overlapping date which will determine whether concatenating both dataset is a reasonable decision. 

In [95]:
#   Convert the invoice string to a date and time and format the values. 
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%d/%m/%y %H:%M')
df2['InvoiceDate'] = pd.to_datetime(df2['InvoiceDate'], format='%d/%m/%y %H:%M')

# Print the start and end values for each dataset using Invoice Date only. 
print(df['InvoiceDate'].min(), df['InvoiceDate'].max())
print(df2['InvoiceDate'].min(), df2['InvoiceDate'].max())

2009-12-01 07:45:00 2010-12-09 20:01:00
2010-12-01 08:26:00 2011-12-09 12:50:00


The above code cell shows the results of the minimum and maximum dates found in both datasets. After being converted form strings and formatted, the results show that there is an eight day overlap between the end of the first dataset and the start of the second:

(df(09.12.2010) - df2(01.12.2010))

With both datasets stretching over a combined 738 days. Removing 8 days of overlap would still leave a comprehenisive dataset. 

In [92]:
# Initilise combined - joining df and df2 together, ignoring indexing to reduce conflicts. 
combined = pd.concat([df, df2], ignore_index=True)
# Re-initlise combined after dropping any duplicate values found within Invoice or Stock code. 
combined = combined.drop_duplicates(subset=['Invoice', 'StockCode'])

#Print the combined dataset shape.
print(combined.shape)
#Print the number of unique ID's
print(combined['Customer ID'].nunique())
# Print the ratio of Custuomer ID's to transactions.
print(len(combined) / combined['Customer ID'].nunique())

(787193, 8)
5942
132.47946819252778
